Combine the dataframes with information distribution of each metric per band (one point per peak group) for different scan

## Step 1 – Imports and location of the data

Each scan of the observation `EDD_2023-03-11T14_58_01` has its own sub-folder (`scan_XXX_PY_NDZ`), and inside each one the pipeline wrote a file called `metrics_per_peak_<scan>.csv`.
Here we import the libraries and define `BASE_DIR` as the folder that contains this notebook (and the scan sub-folders).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()   # .../Project_realignment/EDD_2023-03-11T14_58_01
OBS_ID = BASE_DIR.name  # 'EDD_2023-03-11T14_58_01'
print(BASE_DIR)

/home/chavango/Master/SKAMPI_DATA/work_with_the_dataset/Detrending/peak_analysis/Project_realignment/EDD_2023-03-11T14_58_01


## Step 2 – Find every `metrics_per_peak` file

Instead of typing the file names by hand, we search with a glob pattern `scan_*/metrics_per_peak_*.csv`.
This way, if new scans are added to the folder later, they are picked up automatically. The list is sorted so the scans are always read in the same order.

In [2]:
files = sorted(BASE_DIR.glob("scan_*/metrics_per_peak_*.csv"))

for f in files:
    print(f.relative_to(BASE_DIR))
print(f"\n{len(files)} files found")

scan_000_P1_ND0/metrics_per_peak_scan_000_P1_ND0.csv
scan_001_P0_ND0/metrics_per_peak_scan_001_P0_ND0.csv
scan_001_P1_ND0/metrics_per_peak_scan_001_P1_ND0.csv

3 files found


## Step 3 – Read each file and tag it with its origin

Each CSV is read into its own DataFrame. Before combining, we add a few columns that identify where every row came from:

* `scan_id` – the name of the scan folder (e.g. `scan_001_P0_ND0`), which encodes scan number, polarisation (`P0`/`P1`) and noise diode state (`ND0`/`ND1`).
* `observation` – the observation folder name, useful if later you combine several observations.

This matters because `peak_group` restarts at 0 in every scan: peak 0 of `scan_000_P1_ND0` and peak 0 of `scan_001_P0_ND0` are **not** the same emitter. Without the tag, the rows could not be told apart after concatenation.

In [3]:
frames = []
for f in files:
    df = pd.read_csv(f)
    df.insert(0, "scan_id", f.parent.name)
    df.insert(0, "observation", OBS_ID)
    frames.append(df)
    print(f"{f.parent.name:<18} rows={len(df):>4}  cols={df.shape[1]}")

scan_000_P1_ND0    rows=  80  cols=27
scan_001_P0_ND0    rows= 140  cols=27
scan_001_P1_ND0    rows=  82  cols=27


## Step 4 – Check the files are compatible

`pd.concat` aligns columns by name. If one scan had a missing or renamed column, the result would silently get `NaN` in that column. So before concatenating, we check that all files have exactly the same columns.

In [4]:
reference_cols = list(frames[0].columns)
for f, df in zip(files, frames):
    if list(df.columns) != reference_cols:
        missing = set(reference_cols) - set(df.columns)
        extra = set(df.columns) - set(reference_cols)
        raise ValueError(f"{f.parent.name}: missing={missing}, extra={extra}")
print("All files share the same", len(reference_cols), "columns")

All files share the same 27 columns


## Step 5 – Concatenate into a single DataFrame

`pd.concat` stacks the DataFrames vertically (one below the other). `ignore_index=True` creates a fresh 0..N-1 index, because the original indices repeat between scans.

We also create `peak_uid` (`scan_id` + `peak_group`), a key that is unique across the whole combined table.

In [5]:
metrics_all = pd.concat(frames, ignore_index=True)

metrics_all.insert(2, "peak_uid",
                   metrics_all["scan_id"] + "_peak" + metrics_all["peak_group"].astype(str))

assert metrics_all["peak_uid"].is_unique
assert len(metrics_all) == sum(len(df) for df in frames)

print(metrics_all.shape)
metrics_all.head()

(302, 28)


,observation,scan_id,peak_uid,peak_group,scan,scan_type,polarization,f_ref_kHz,band,detection_level,...,A,S,B99_cv,FWHM_cv,B10_cv,B20_cv,B30_cv,RBW_cv,A_cv,S_cv
0,EDD_2023-03-11T14_58_01,scan_000_P1_ND0,scan_000_P1_ND0_peak0,0,scan/000/P1_ND0,Azimuth-Scan,Vertical,1.819628e+06,Cellular GSM Downlink,high,...,1.302107e+13,1.294880,0.029179,0.017074,0.061720,0.036104,0.045759,0.036701,0.006220,0.035870
1,EDD_2023-03-11T14_58_01,scan_000_P1_ND0,scan_000_P1_ND0_peak1,1,scan/000/P1_ND0,Azimuth-Scan,Vertical,1.846865e+06,Cellular GSM Downlink,low,...,2.298056e+12,-0.340238,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EDD_2023-03-11T14_58_01,scan_000_P1_ND0,scan_000_P1_ND0_peak2,2,scan/000/P1_ND0,Azimuth-Scan,Vertical,1.873568e+06,Cellular GSM Downlink,high,...,1.805238e+11,0.079344,0.657772,0.261239,0.607883,0.595564,0.643853,0.881668,0.383572,3.480078
3,EDD_2023-03-11T14_58_01,scan_000_P1_ND0,scan_000_P1_ND0_peak3,3,scan/000/P1_ND0,Azimuth-Scan,Vertical,1.875917e+06,Cellular GSM Downlink,high,...,1.701464e+11,-1.276138,0.573853,0.418998,0.634854,0.662728,0.529749,0.651366,0.537620,1.393426
4,EDD_2023-03-11T14_58_01,scan_000_P1_ND0,scan_000_P1_ND0_peak4,4,scan/000/P1_ND0,Azimuth-Scan,Vertical,1.877920e+06,Cellular GSM Downlink,low,...,6.832910e+10,-0.339187,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 6 – Quick sanity checks

* Number of peaks per scan, split by detection level (`high` / `low`).
* Missing values per column: peaks with `n_bins == 1` have no `fc_std` and no `*_cv` (coefficient of variation), because a spread cannot be computed from a single time bin. This is expected, not a reading error.

In [6]:
display(pd.crosstab(metrics_all["scan_id"], metrics_all["detection_level"], margins=True))

na = metrics_all.isna().sum()
na[na > 0]

detection_level,high,low,All
scan_id,,,
scan_000_P1_ND0,12,68,80
scan_001_P0_ND0,15,125,140
scan_001_P1_ND0,9,73,82
All,36,266,302


fc_std     118
B99_cv     118
FWHM_cv    118
B10_cv     118
B20_cv     118
B30_cv     118
RBW_cv     118
A_cv       118
S_cv       118
dtype: int64

In [7]:
# Confirm the NaNs in the variability columns come from single-bin peaks
cv_cols = ["fc_std"] + [c for c in metrics_all.columns if c.endswith("_cv")]
single_bin = metrics_all["n_bins"] == 1
print("rows with NaN in fc_std:", metrics_all["fc_std"].isna().sum())
print("of which n_bins == 1   :", (metrics_all["fc_std"].isna() & single_bin).sum())

rows with NaN in fc_std: 118
of which n_bins == 1   : 118


## Step 7 – First look at the combined data

Now that all scans are in one table, comparisons across scans are a single `groupby`. Here: number of peaks per band and scan, and the median of the main metrics per band and scan.

In [8]:
pd.crosstab(metrics_all["band"], metrics_all["scan_id"], margins=True)

scan_id,scan_000_P1_ND0,scan_001_P0_ND0,scan_001_P1_ND0,All
band,,,,
Air Traffic Radar,5,34,8,47
Cellular GSM Downlink,5,2,5,12
DECT Phones,2,0,1,3
Gap: EESS to IMT-2300,2,3,2,7
Gap: GSM DL to GSM UL,0,1,0,1
Gap: ISM to LTE 2600,4,3,4,11
Gap: LTE 2600 to Radar,0,2,0,2
IMT 2300 (4G/5G),15,22,18,55
LTE / 5G (2600 band),8,20,10,38


In [9]:
metric_cols = ["B99", "FWHM", "B10", "B20", "B30", "RBW", "A", "S"]

summary = (metrics_all
           .groupby(["band", "scan_id"])[metric_cols]
           .median())
summary

B99         FWHM  \
band                    scan_id                                     
Air Traffic Radar       scan_000_P1_ND0   507.354736    58.318269   
                        scan_001_P0_ND0   567.436218    58.269797   
                        scan_001_P1_ND0   453.948975    62.425695   
Cellular GSM Downlink   scan_000_P1_ND0  1548.767090    73.946396   
                        scan_001_P0_ND0  4799.842834   229.360394   
                        scan_001_P1_ND0  1375.198364    48.191298   
DECT Phones             scan_000_P1_ND0  7049.560547   238.670934   
                        scan_001_P1_ND0  7823.944092   855.523961   
Gap: EESS to IMT-2300   scan_000_P1_ND0   974.655151   130.898316   
                        scan_001_P0_ND0   307.083130    73.357840   
                        scan_001_P1_ND0   273.704529    47.189925   
Gap: GSM DL to GSM UL   scan_001_P0_ND0  6622.314453   129.145386   
Gap: ISM to LTE 2600    scan_000_P1_ND0  2022.743225   106.225276   
                        scan_001_P0_ND0  1976.013184    96.336763   
                        scan_001_P1_ND0  1455.307007    84.780001   
Gap: LTE 2600 to Radar  scan_001_P0_ND0   347.137451    55.008441   
IMT 2300 (4G/5G)        scan_000_P1_ND0   587.463379    74.482423   
                        scan_001_P0_ND0   687.599182    56.180074   
                        scan_001_P1_ND0  1001.358032    64.120845   
LTE / 5G (2600 band)    scan_000_P1_ND0   720.977783    47.946795   
                        scan_001_P0_ND0   587.463379    48.809561   
                        scan_001_P1_ND0   654.220581    58.392554   
Mobile Satellite        scan_000_P1_ND0  2136.230469   365.233273   
                        scan_001_P0_ND0  3578.186035   368.513260   
                        scan_001_P1_ND0  3150.939941    65.771710   
Space Research / EESS   scan_000_P1_ND0   520.706177    53.252937   
                        scan_001_P0_ND0   801.086426    66.214057   
                        scan_001_P1_ND0   720.977783    56.316539   
UMTS Downlink           scan_000_P1_ND0  4045.486450  1405.023872   
                        scan_001_P0_ND0  4058.837891   741.417774   
                        scan_001_P1_ND0  4005.432129  1012.934609   
UMTS Uplink             scan_001_P1_ND0  4192.352295   220.283281   
Wi-Fi / Bluetooth (ISM) scan_000_P1_ND0   534.057617    55.180530   
                        scan_001_P0_ND0   413.894653    52.852997   
                        scan_001_P1_ND0   640.869141    69.201079   

                                                 B10          B20  \
band                    scan_id                                     
Air Traffic Radar       scan_000_P1_ND0   101.499660   581.575543   
                        scan_001_P0_ND0   254.001615   604.598857   
                        scan_001_P1_ND0   156.194279   456.974052   
Cellular GSM Downlink   scan_000_P1_ND0   317.816752   795.879092   
                        scan_001_P0_ND0   663.129076  4185.831304   
                        scan_001_P1_ND0   195.040712   991.632134   
DECT Phones             scan_000_P1_ND0  4227.628409  7183.773416   
                        scan_001_P1_ND0  5093.409181  7993.085088   
Gap: EESS to IMT-2300   scan_000_P1_ND0   469.334132   952.740759   
                        scan_001_P0_ND0   214.829248   379.745420   
                        scan_001_P1_ND0   205.914293   337.912308   
Gap: GSM DL to GSM UL   scan_001_P0_ND0  2147.518940  6377.729844   
Gap: ISM to LTE 2600    scan_000_P1_ND0  1074.269175  1840.407089   
                        scan_001_P0_ND0  1624.309109  2064.676001   
                        scan_001_P1_ND0   661.855660  1511.787816   
Gap: LTE 2600 to Radar  scan_001_P0_ND0   126.605325   416.885312   
IMT 2300 (4G/5G)        scan_000_P1_ND0   292.302931   663.308180   
                        scan_001_P0_ND0   208.432052   605.939865   
                        scan_001_P1_ND0   461.573152  1039.727768   
LTE / 5G (2600 band)    scan_000_P1_ND0   394.

## Step 8 – Save the combined table

The result is written to `metrics_per_peak_combined_<observation>.csv` in this folder, so other notebooks can load it directly with `pd.read_csv` without repeating these steps.

In [10]:
out_file = BASE_DIR / f"metrics_per_peak_combined_{OBS_ID}.csv"
metrics_all.to_csv(out_file, index=False)
print("Saved:", out_file.name, metrics_all.shape)

Saved: metrics_per_peak_combined_EDD_2023-03-11T14_58_01.csv (302, 28)
